In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
  model_id,
  dtype=torch.bfloat16
).to("cuda").eval()

messages = [
  {
    "role": "user",
    "content": "Transformer의 Attention을 설명해줘"
  }
]

inputs = tokenizer.apply_chat_template(
  messages,
  tokenize=True,
  add_generation_prompt=True,
  enable_thinking=False,
  return_tensors="pt"
).to("cuda")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [3]:
type(tokenizer), type(model)

(transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer,
 transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM)

In [4]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
        (post_attention_layer

In [5]:
f"{sum([p.numel() for p in model.parameters()]):,}"

'1,720,574,976'

In [6]:
GiB = 2**(10 + 10 + 10)
f"{GiB:,}"

'1,073,741,824'

In [7]:
12*GiB / (1000**3 * 4)

3.221225472

In [8]:
f"{(1000**3 * 4):,}"

'4,000,000,000'

In [9]:
f"{sum([p.numel() for p in model.parameters()])*(2 if next(model.parameters()).dtype in (torch.bfloat16, torch.float16) else 4)/GiB:,.2f}GB"

'3.20GB'

In [10]:
inputs

{'input_ids': tensor([[151644,    872,    198,  46358,  20401,  62020,  17877, 133828,  33883,
         144131, 151645,    198, 151644,  77091,    198, 151667,    271, 151668,
            271]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}

In [11]:
tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

['<|im_start|>',
 'user',
 'Ċ',
 'Transformer',
 'ìĿĺ',
 'ĠAttention',
 'ìĿĦ',
 'ĠìĦ¤ëªħ',
 'íķ´',
 'ì¤ĺ',
 '<|im_end|>',
 'Ċ',
 '<|im_start|>',
 'assistant',
 'Ċ',
 '<think>',
 'ĊĊ',
 '</think>',
 'ĊĊ']

In [12]:
tokenizer.decode(
  inputs["input_ids"][0],
  # skip_special_tokens=True
)

'<|im_start|>user\nTransformer의 Attention을 설명해줘<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

In [13]:
type(inputs)

transformers.tokenization_utils_base.BatchEncoding

In [14]:
with torch.inference_mode():
  outputs = model.generate(
    **inputs,
    max_new_tokens=1024
  )

In [15]:
outputs.shape

torch.Size([1, 1043])

In [16]:
inputs["input_ids"][-1]

tensor([151644,    872,    198,  46358,  20401,  62020,  17877, 133828,  33883,
        144131, 151645,    198, 151644,  77091,    198, 151667,    271, 151668,
           271], device='cuda:0')

In [17]:
print(tokenizer.decode(
  outputs[0][inputs["input_ids"].shape[-1]:],
  skip_special_tokens=True
))

Transformer 모델에서 **Attention**은 핵심 기술 중 하나로, 모델이 입력 데이터를 어떻게 처리하고 어떤 정보를 강조하는지 결정하는 데 중요한 역할을 합니다. Attention은 **모델이 입력에 대한 각각의 토큰과의 관계를 파악**하고, 강조할 정보를 선택하는 데 사용됩니다.

---

## 🔍 1. Attention의 기본 개념

### 🧠 **Attention**은 두 가지 주요 유형이 있습니다:

#### 1. **Self-Attention (자기 조정 Attention)**
- **자기 조정**은 모델이 자신의 입력에 대해 정보를 조정하는 방식입니다.
- 이는 **모델이 입력에 대한 관계를 파악**하는 데 사용됩니다.

#### 2. **Cross-Attention (다른 모델의 Attention)**
- 다른 텍스트나 데이터에 대한 관계를 파악하는 방식입니다.
- 예: 텍스트 생성 시 다른 텍스트에 대한 정보를 참조하는 경우.

---

## 🧠 2. Self-Attention의 구조

Self-Attention은 입력 벡터 $ Q $, $ K $, $ V $로 구성됩니다.

### 📌 1. Query (Q), Key (K), Value (V) 생성

- 입력 벡터 $ X $를 $ Q, K, V $로 분리합니다.
- $ Q = X \cdot \text{Query Matrix} $
- $ K = X \cdot \text{Key Matrix} $
- $ V = X \cdot \text{Value Matrix} $

### 📌 2. Dot Product (점 곱) 계산

- $ \text{Attention}(Q, K, V) = \frac{Q \cdot K^T}{\sqrt{d_k}} $
- 여기서 $ d_k $는 Key의 차원입니다.

### 📌 3. Attention 결과를 Value로 사용

- $ \text{Attention}(Q, K, V) $는 $ V $로 사용됩니다.
- 결과는 **각 토큰에 대한 정보를 강조**하는 방식으로 처